In [ ]:
import csv
import sys
import numpy as np
import os
csv.field_size_limit(sys.maxsize)
import ast
import random
import matplotlib.pyplot as plt
import time
import pickle
import gc
from gpcam import GPOptimizer
from datetime import datetime
import shutil
from functools import partial
import dask
from dask.distributed import Client
import os
import time
from gpAMRutils import *

## Tuning options to consider
- Filtering data by block or as a function of the global data?
- What fraction of candidates should be used, > mean SD? top 10%, top n candidates?
- How will data noise be estimated and set?
- RAM: submitting ask several times means that $\kappa \in \mathbb{R}^{12000 \times 12000}$ is possibly stored many times at once. At the same time the interpolator in the kernel stores temporarily 2(3) matrices of shape (len(x_data)^2)

## SETTINGS

README:
- The advection example grid is 32 by 64, 64 x 128, 128 x 256
- Each block is 32 x 32, making for 2/8/32 blocks
- Therefore, "./allocateCPUsADVECTION.sh nodes workers" (usually workers = 4 * nodes, except for 32x64, there it's just '1 2')
- And then start the Dask scheduler and workers: "./launch-dask-moduleCPU_ADVECTION.sh nodes workers" (same numbers as in allocate)

In [ ]:
#HPC SETTINGS:
number_of_workers = 32 #match dask launch script and allocation (allocate_CPUs.sh 1 2)
#number_of_threads = 32 #currently not used

#BLOCKING
no_blocks_x = 8
no_blocks_y = 4
pad = 8 #>= 1


#DATA CURATION
normalizing = False #y_data in [0,1]
filtering = False #remove data below a threshold
tol_ratio = 1e-2 #remove data < tol_ratio * max(data)

#REFINEMENT LEVEL
refinement_res = 2 #refinement level


#PLOTTING
plotting = True #plot posterior mean and var
plot_resx = 100
plot_resy =  50
plot_every = 2 #iterations


#FILE HANDLING
#filename = "plot.nx64.2d.AMRLevel.hdf5"
#filename = "plot.nx128.2d.AMRLevel.hdf5"
filename = "plot.nx256.2d.AMRLevel.hdf5"

index = "component0"
chombo_path = "./ChomboOut/"
gpcam_path = "./gpCAMOut/"
rename_file = True #if True, Chombo file will be read and then renamed for bookkeeping; if used, turn delete_file = False
delete_file = True #if true, rename_file is ignored

#GP setup
percentile = 90 ##of all candidates considered, return the top percentile
uncertainty_cutoff = 0.0015
init_hyperparameters = np.array([0.000001, 30., 20.]) #signal var, length scale, gradient sensitivity
hyperparameter_bounds = np.array([[0.001, 10.],
                                  [0.1, 50.],
                                  [0.01, 50.]
                                 ])
match_hps = True
noise = 0.0001 #variance


#####TEST MODE###################################
#################################################
#filename = "TESTadv.hdf5"
#filename = "plot.nx256.2d.test.hdf5"
#rename_file = False #if True, Chombo file will be read and then renamed for bookkeeping; if used, turn delete_file = False
#delete_file = False #if True, rename_file is ignored
#################################################




In [ ]:
#check if Chombo file already exists
if os.path.exists(chombo_path+"ready.txt") and os.path.exists(chombo_path+filename):
    print("The data file ", chombo_path+filename," already exists in the repo. Should I delete it? y/n")
    dec = input()
    if dec  == "y":
            print("File removed ...")
            os.remove(chombo_path+filename)
            os.remove(chombo_path+"ready.txt")
    elif dec == "n": print("file not deleted")
    else: print("This was not a viable option, run the cell again!")

In [ ]:
###TEST DATA AND CHECK FORMATTING
data = read_fileIII(chombo_path, filename, index, rename = False, delete=False, n_sub_x = no_blocks_x, n_sub_y = no_blocks_y, pad_x=pad, pad_y=pad)
print(data)
print("global data")
plt.scatter(data["global_coordinates"][:,0], data["global_coordinates"][:,1],c = data["global_funcvalues"])
plt.show()
for n in data["local_data"]:
      print("local data", n)
      plt.scatter(data["local_data"][n]["points"][:,0], data["local_data"][n]["points"][:,1], c = data["local_data"][n]["values"])
      plt.show()


refinement_candidates = make_refinement_candidates(data, refinement_level=refinement_res)
global_refinement_grid = refinement_candidates["global_candidates"]
local_refinement_grids = refinement_candidates["local_candidates"]
print("global_refinement_grid")
plt.scatter(global_refinement_grid[:,0], global_refinement_grid[:,1], s = 0.1)
plt.show()
for n in data["local_data"]:
     print("local refinement grid ", n)
     plt.scatter(local_refinement_grids[n][:,0], local_refinement_grids[n][:,1], s = 0.1)
plt.show()

## Dask Client

In [ ]:
scheduler_file = os.path.join(os.environ["SCRATCH"], "scheduler_filegpAMR.json")
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 

client = init_client(scheduler_file, number_of_workers)

In [ ]:
client

## GP set-up

In [ ]:
from scipy.interpolate import griddata
from gpcam.kernels import *
from scipy.interpolate import RBFInterpolator
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve
from scipy.spatial.distance import cdist

from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix, identity

from scipy.interpolate import CloughTocher2DInterpolator, NearestNDInterpolator
def padded_ct(points, values, pad_frac=0.1, n_ghost=60):
    lo, hi = points.min(0), points.max(0)
    span = hi - lo
    plo, phi = lo - pad_frac*span, hi + pad_frac*span
    t = np.linspace(0, 1, n_ghost)
    ring = np.vstack([
        np.c_[plo[0] + t*(phi[0]-plo[0]), np.full_like(t, plo[1])],
        np.c_[plo[0] + t*(phi[0]-plo[0]), np.full_like(t, phi[1])],
        np.c_[np.full_like(t, plo[0]), plo[1] + t*(phi[1]-plo[1])],
        np.c_[np.full_like(t, phi[0]), plo[1] + t*(phi[1]-plo[1])],
    ])
    ghost = NearestNDInterpolator(points, values)(ring)   # Neumann-style extension
    P = np.vstack([points, ring])
    V = np.concatenate([values, ghost])
    return CloughTocher2DInterpolator(P, V, fill_value=0.), span

def int_obj(x_data, y_data):
    rbf, span = padded_ct(x_data, y_data)
    return  rbf, span

def interpolator(rbf, x):
    return rbf(x)

def interpolator_grad(rbf, span, x):
    h = 1e-3 * span.min()
    ex, ey = np.array([h, 0]), np.array([0, h])
    gx = (rbf(x + ex) - rbf(x - ex)) / (2*h)
    gy = (rbf(x + ey) - rbf(x - ey)) / (2*h)
    norm = np.hypot(gx, gy)
    return norm


def kernelPDEII(x1, x2, hps, args):
    """
    Non-stationary Gibbs kernel (gpAMR Eq. 4) with the length-scale field
    conditioned on the PDE-solution gradient (Eq. 5):

        sigma_f(x) = |q(x)|                          # signal std  <- solution VALUE
        ell(x)     = ell0 / (1 + beta |grad q(x)|)   # length scale <- solution GRADIENT

    hps[0] : signal-variance scale coefficient
    hps[1] : ell0, the smooth-region (maximum) length scale
    hps[2] : beta, gradient sensitivity of the length scale  (NEW)
    """
    x_data = args["x_data"]
    y_data = args["y_data"]
    D = x1.shape[1]
    rbf = args["rbf"]
    span = args["span"]

    # signal variance from the solution VALUE: sigma_f(x) = |q(x)|
    #sf1 = np.abs(rbf(x1))
    #sf2 = np.abs(rbf(x2))
    #rbf, span = int_obj(x_data, y_data)
    sf1 = abs(interpolator(rbf, x1))
    sf2 = abs(interpolator(rbf, x2))

    # length scale from the solution GRADIENT: small where |grad q| is large
    g1 = interpolator_grad(rbf, span, x1)
    g2 = interpolator_grad(rbf, span, x2)
    
    gmag1 = np.abs(g1) if g1.ndim == 1 else np.linalg.norm(g1, axis=1)
    gmag2 = np.abs(g2) if g2.ndim == 1 else np.linalg.norm(g2, axis=1)
    del rbf

    ell0, beta = hps[1], hps[2]
    l1 = ell0 / (1.0 + beta * gmag1)      # contracts where the field is steep
    l2 = ell0 / (1.0 + beta * gmag2)

    # Gibbs non-stationary SE kernel (Eq. 4, isotropic ell)
    denom = (l1**2)[:, None] + (l2**2)[None, :]          # ell(x1)^2 + ell(x2)^2
    prefactor = (2.0 * np.outer(l1, l2) / denom) ** (D / 2.0)
    d2 = get_distance_matrix(x1, x2) ** 2
    gibbs = prefactor * np.exp(-d2 / denom)

    #return hps[0] * np.outer(sf1, sf2) * gibbs
    return hps[0] * gibbs

def meanf(x,hps):
    return np.zeros(len(x))

def acq_func(x, gp):
    x = np.asarray(x)
    return np.sqrt(gp.posterior_covariance(x, variance_only=True)["v(x)"])

In [ ]:
data = read_fileIII(chombo_path, filename, index, rename = rename_file, delete=delete_file, n_sub_x = no_blocks_x, n_sub_y = no_blocks_y, pad_x=pad, pad_y=pad)
if normalizing: data = normalize_data(data)
if filtering: data = filter_all_data(data, tol_ratio)

refinement_candidates = make_refinement_candidates(data, refinement_level=refinement_res)
global_refinement_grid = refinement_candidates["global_candidates"]
local_refinement_grids = refinement_candidates["local_candidates"]


rbf, span = int_obj(data["global_coordinates"], data["global_funcvalues"])
GPs = {}
candidate_pools = {}
block_domains = {}
kernels = {}
print("Initializing GPs")
for blockID in data["local_data"]:
    kernels[blockID] = kernelPDEII
    x_data = data["local_data"][blockID]["points"]
    y_data = data["local_data"][blockID]["values"]
    print("min/max y: ", np.min(y_data),np.max(y_data))
    print("dataset size: ", x_data.shape)
    block_init_hps = init_hyperparameters.copy()
    block_init_hps[0] = max(np.var(y_data),hyperparameter_bounds[0,0])
    GPs[blockID] = GPOptimizer(x_data, y_data, noise_variances=np.ones(y_data.shape) * noise,
                          kernel_function=kernels[blockID],
                          prior_mean_function=meanf,
                          init_hyperparameters = block_init_hps,
                          args={"active": True, "x_data": data["global_coordinates"], "y_data": data["global_funcvalues"], "rbf": rbf, "span": span}, linalg_mode = "CholInv")

    #define refinement res
    xmin = data["local_data"][blockID]["bounds"]["interior_cols"][0]
    xmax = data["local_data"][blockID]["bounds"]["interior_cols"][1]
    ymin = data["local_data"][blockID]["bounds"]["interior_rows"][0]
    ymax = data["local_data"][blockID]["bounds"]["interior_rows"][1]    
    candidate_pools[blockID] = [array for array in local_refinement_grids[blockID]]
    if not valid(candidate_pools[blockID]): raise Exception("Invalid candidate pool in blockID ", blockID)
    block_domains[blockID] = np.array([[xmin,xmax],
                                       [ymin,ymax]])
    print("blockdomains:", block_domains[blockID])
    print("")
print("Initialization done!")

all_candidates = global_refinement_grid
if not valid(all_candidates): raise Exception("Duplicates in global candidate pool")


GPs = client.scatter(GPs, broadcast=False, direct=True)


#Training
print("Initial training...")
LL = {}
if match_hps: 
    for entry in GPs: LL[entry] = np.std(GPs[entry].result().y_data)
    maxLL = max(LL, key=lambda k: LL[k])
    print("Matching hps. Best hps: ", maxLL)
    _train = train(client, hyperparameter_bounds, GPs[maxLL],  max_iter = 1000, method = "mcmc").result()
    for entry in GPs: 
        if entry == maxLL: continue
        set_hps(client, GPs[entry], GPs[maxLL].result().hyperparameters)
else:
    train_futures = []
    for blockID in data["local_data"]: train_futures.append(train(client, hyperparameter_bounds, GPs[blockID],  max_iter = 1000, method = "mcmc"))
    client.gather(train_futures)
print("Initial training done!")








# print("Initial training...")
# train_futures = []
# for blockID in data["local_data"]: 
#     train_futures.append(train(client, hyperparameter_bounds, GPs[blockID],  max_iter = 500, method = "mcmc"))
# client.gather(train_futures)
print("Initial training done!")
for entry in GPs:print("hyperparameters GP", entry," : ",GPs[entry].result().hyperparameters)

###################################################
###################################################
iteration_counter = 0
print("#######################")
print("start gpAMR iteration: ")
print("#######################")
suggestion_history = []
plot_counter = plot_every-1
training_at = [2,5,10,20]
while True:
    iteration_counter += 1
    print("")
    print("")
    print("++++++++++++++++++++++++++++++++++")
    print("start gpAMR iteration: ", iteration_counter)
    print("++++++++++++++++++++++++++++++++++")
    
    res = []
    #ASKING FOR SUGGESTIONS
    print("Asking for new suggestions")
    for blockID in data["local_data"]:
        if not GPs[blockID].result().args["active"]: continue
        print("        Asking GP ",blockID, " with ",len(GPs[blockID].result().x_data)," data points, for suggestions")
        candidate_pool = candidate_pools[blockID]
        #candidates = list(chunks(candidate_pool, number_of_threads))
        print("        ask for new suggestions.... Candidates  considered: ", len(candidate_pool))
        #for chunk in candidates: res.append(ask(client, chunk, GPs[ID], len(chunk), acq_func)) ###FAST BUT RAM INEFFICIENT
        #print(np.asarray(candidate_pool))
        res.append(ask(client, candidate_pool, GPs[blockID], len(candidate_pool), acq_func)) ###SLOWER BUT RAM EFFICIENT
    new = client.gather(res)

    print("    All GP agents reported suggestions...concatenating")
    SD = np.concatenate([x["f_a(x)"] for x in new])
    new = np.vstack([x["x"] for x in new])
    sorted_indices = np.argsort(SD)[::-1]
    SD = SD[sorted_indices]
    new = new[sorted_indices]
    uncertainty_tol = np.percentile(SD, percentile)
    plt.plot(SD)
    print("uncertainty tol: ", uncertainty_tol, "uncertainty cutoff: ", uncertainty_cutoff)
    plt.show()
    non_zero_ind = np.where(SD > max(uncertainty_tol,uncertainty_cutoff))
    if not valid(new): raise Exception("Non-Unique suggestions")
    suggestions = new[non_zero_ind]
    print("len(suggestions): ", len(suggestions))
    print("    Suggestions calculated, len:", len(suggestions))
    print("    Write suggestions for Chombo iteration... ", iteration_counter)
    #################################
    #suggestions = np.round(suggestions)
    #################################
    if not valid(suggestions): raise Exception("Non-Unique suggestions")
    write_file(gpcam_path, chombo_path, suggestions) ##send to data generator (simulation)
    print("    Suggestions written!")
    suggestion_history.append(suggestions)
    

    #PLOTTING
    plot_counter+=1
    if plotting and plot_counter==plot_every:
        plot_counter=0
        print("Generating plots...")
        mean_futures = []
        cov_futures = []
        for blockID in data["local_data"]:
            if not GPs[blockID].result().args["active"]: continue 
            x_plot = GPOptimizer.make_2d_x_pred(block_domains[blockID][0],block_domains[blockID][1],resx=plot_resx,resy=plot_resy)
            mean_futures.append(posterior_mean(client, x_plot,GPs[blockID]))
            cov_futures.append(posterior_covariance(client, x_plot,GPs[blockID]))
        means_tmp = client.gather(mean_futures)
        means = np.concatenate([mean["m(x)"] for mean in means_tmp])
        cov_tmp = client.gather(cov_futures)
        stds = np.concatenate([np.sqrt(cov["v(x)"]) for cov in cov_tmp])
        x_pred = np.vstack([mean["x_pred"] for mean in means_tmp])
        
        print("global data and suggestions:")
        plt.figure(figsize=(20,5))
        a = plt.scatter(data["global_coordinates"][:,0], data["global_coordinates"][:,1], c=data["global_funcvalues"], alpha=.2)
        plt.scatter(suggestions[:,0], suggestions[:,1], s=0.1, c='black', alpha=1.)
        plt.colorbar(a)
        plt.show()

        print("global mean and suggestions:")
        plt.figure(figsize=(20,5))
        plot2d(x_pred[:,0], x_pred[:,1], means, suggestions=suggestions, title= "mean and suggestions",filename=gpcam_path+"mean" + str(iteration_counter).zfill(4))
        
        print("global std and suggestions:")
        plot2d(x_pred[:,0], x_pred[:,1], stds, suggestions=suggestions, title= "std and suggestions", filename=gpcam_path+"std" + str(iteration_counter).zfill(4))

        ##write image to disc
        print("Write to file...")
        print("POSTERIOR MEAN")
        plt.figure(figsize=(20,5))
        a = plt.scatter(x_pred[:,0],x_pred[:,1],c = means, alpha=1.)
        #plt.scatter(suggestions[:,0], suggestions[:,1], s=0.1, c='black', alpha=1.0)
        plt.xlim(data["domain"][0,0], data["domain"][0,1])
        plt.ylim(data["domain"][1,0], data["domain"][1,1])
        plt.colorbar(a)
        plt.savefig(gpcam_path+"mean_sugg" + str(iteration_counter).zfill(4))
        plt.show()
        ##write image to disc
        print("POSTERIOR SD")
        plt.figure(figsize=(20,5))
        a = plt.scatter(x_pred[:,0],x_pred[:,1], c = stds, alpha=0.5)
        #plt.scatter(suggestions[:,0], suggestions[:,1], s=0.1, c='black', alpha=0.3)
        plt.xlim(data["domain"][0,0], data["domain"][0,1])
        plt.ylim(data["domain"][1,0], data["domain"][1,1])
        plt.colorbar(a)
        plt.savefig(gpcam_path+"std_sugg" + str(iteration_counter).zfill(4))
        plt.show()
        print("Done plotting for this iteration!")
    
    

    print("Reading Chombo file. Iteration: ", iteration_counter)
    data = read_fileIII(chombo_path, filename, index, rename = rename_file, delete=delete_file, n_sub_x = no_blocks_x, n_sub_y = no_blocks_y, pad_x=pad, pad_y=pad)
    rbf, span = int_obj(data["global_coordinates"], data["global_funcvalues"])
    if normalizing: data = normalize_data(data)
    if filtering: data = filter_all_data(data, tol_ratio)
    #rbf = WendlandRBF(data["global_coordinates"], data["global_funcvalues"], epsilon=epsilon)
    print("global dataset size: ", len(data["global_coordinates"]))
    print("filter tol: ", tol_ratio * np.max(data["global_funcvalues"]))
    print("Received data:")
    plt.figure(figsize=(20,5))
    a = plt.scatter(data["global_coordinates"][:,0], data["global_coordinates"][:,1], c=data["global_funcvalues"], alpha=.2)
    plt.colorbar(a)
    plt.show()
    print("done!")

    
    #UPDATE GPs
    print("Updating GP agents...")
    update_futures = [] 
    for blockID in data["local_data"]:
        x_data = data["local_data"][blockID]["points"]
        y_data = data["local_data"][blockID]["values"]
        set_args(client, GPs[blockID], {"active": True, "x_data": data["global_coordinates"], "y_data": data["global_funcvalues"], "rbf": rbf, "span": span})
        update_futures.append(tell(client, x_data, y_data, np.ones(y_data.shape) * noise, GPs[blockID]))
        new_hyperparameters = GPs[blockID].result().hyperparameters.copy()
        new_hyperparameters[0] = max(np.var(y_data),hyperparameter_bounds[0,0])
        set_hps(client, GPs[blockID], new_hyperparameters)
            
    client.gather(update_futures)
    print("Updating GP agents done!")

    #TRAINING
    if iteration_counter in training_at:
        print("Training...")
        train_futures = []
        for blockID in data["local_data"]: 
            if not GPs[blockID].result().args["active"]: continue
            train_futures.append(train(client, hyperparameter_bounds, GPs[blockID],  max_iter = 100, method = "mcmc"))
        client.gather(train_futures)
        print("Training done!")
    LL = {}
    if match_hps:
        for entry in GPs: LL[entry] = np.std(GPs[entry].result().y_data)
        maxLL = max(LL, key=lambda k: LL[k])
        for entry in GPs: set_hps(client, GPs[entry], GPs[maxLL].result().hyperparameters.copy())
    print("++++++++++++++++++++++++++++++++++")